# Comparative Analysis of Traditional and Transformer-Based Models for Sentiment Classification

## Overview
This project compares a traditional machine learning approach with a transformer-based model for sentiment classification. The goal is to evaluate differences in performance, contextual understanding, and overall suitability for natural language processing tasks.

## Research Question
How do traditional machine learning models compare to transformer-based models in sentiment classification performance and contextual understanding?

In [6]:
!pip install datasets transformers torch scikit-learn pandas

import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import pipeline

In [11]:
#Load Dataset
dataset = load_dataset("imdb")

data = dataset["train"].shuffle(seed=42).select(range(2000))

texts = data["text"]
labels = data["label"]

df = pd.DataFrame({"text": texts, "label": labels})
df.head()

,text,label
0,There is no relation at all between Fortier an...,1
1,This movie is a great. The plot is very true t...,1
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0
3,In the process of trying to establish the audi...,1
4,"Yeh, I know -- you're quivering with excitemen...",0


## Dataset
The dataset used is the IMDB movie review dataset from the Hugging Face datasets library. Reviews are labeled as either positive or negative sentiment.

For this project, a subset of the training split dataset is used to compare a traditional machine learning model with a transformer-based model under the same task setting.

## Traditional Machine Learning Model
The first model uses TF-IDF vectorization with Logistic Regression. This serves as a baseline approach for sentiment classification and represents a common traditional NLP pipeline.

In [12]:
#Train traditional model

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

lr_preds = lr_model.predict(X_test_vec)

lr_accuracy = accuracy_score(y_test, lr_preds)
print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.8175


## Transformer-Based Model
The second model uses a pre-trained DistilBERT sentiment classification pipeline. Transformer models rely on contextual language representations and are expected to perform better on more complex or nuanced text.

In [13]:
#Load transformer pipeline
classifier = pipeline("sentiment-analysis")

#Prepare transformer input
transformer_texts = list(X_test[:200])
transformer_true = list(y_test[:200])

# Run transformer predictions
transformer_outputs = classifier(transformer_texts, truncation=True)
transformer_outputs[:5]

#Convert transformer predictions to labels
transformer_preds = [1 if pred["label"] == "POSITIVE" else 0 for pred in transformer_outputs]

transformer_accuracy = accuracy_score(transformer_true, transformer_preds)
print("DistilBERT Accuracy:", transformer_accuracy)

# Show comparison table
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "DistilBERT"],
    "Accuracy": [lr_accuracy, transformer_accuracy]
})

comparison_df

# Show sample predictions
sample_df = pd.DataFrame({
    "text": transformer_texts[:5],
    "true_label": transformer_true[:5],
    "distilbert_pred": transformer_preds[:5],
    "confidence": [round(pred["score"], 4) for pred in transformer_outputs[:5]]
})

sample_df


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT Accuracy: 0.88


,text,true_label,distilbert_pred,confidence
0,Rarely has such an amazing cast been wasted so...,0,0,0.9998
1,First let me say I am not from the south but I...,1,1,0.9939
2,I'm going to review the 2 films as a whole bec...,1,1,0.9988
3,I had a lot of expectations from this movie an...,0,0,0.9977
4,I believe Shakespeare explained what I just re...,0,1,0.9811


## Results
The Logistic Regression model achieved an accuracy of approximately 0.82, while the transformer-based DistilBERT model achieved an accuracy of approximately 0.88 on the selected subset of the IMDB dataset.

The comparison shows that both approaches perform reasonably well, but the transformer model provides better overall accuracy. This suggests that contextual language representations offer an advantage over traditional feature-based methods in sentiment classification.

## Comparison Analysis
The traditional machine learning model is simpler, faster, and computationally cheaper. However, it relies on TF-IDF representations, which do not capture context, word order, or subtle semantic relationships.

The transformer-based model uses contextual embeddings, which allow it to better interpret sentence meaning and more complex sentiment expressions. Although it requires greater computational resources, it provides stronger performance and more robust predictions.

In [14]:
#Error Analysis
error_df = pd.DataFrame({
    "text": transformer_texts,
    "true_label": transformer_true,
    "distilbert_pred": transformer_preds
})

errors_only = error_df[error_df["true_label"] != error_df["distilbert_pred"]]
errors_only.head(5)

,text,true_label,distilbert_pred
4,I believe Shakespeare explained what I just re...,0,1
9,There's more to offer in the opening of The Od...,1,0
12,"okay, but just plain dumb. Not bad for a horro...",1,0
13,"In today's world of digital fabrication, there...",1,0
16,I couldn't give this film a bad rating or bad ...,0,1


## Error Analysis
Misclassified examples highlight cases where sentiment is ambiguous, mixed, or dependent on broader context. Some reviews contain subtle phrasing, conflicting signals, or unusual wording that make classification difficult.

The error analysis shows that even strong transformer models are not perfect, but it also helps identify the kinds of text that remain challenging for automatic sentiment classification.

## Interpretation
The comparison demonstrates that transformer-based models provide a stronger approach to sentiment classification than traditional machine learning methods. While Logistic Regression offers a useful and efficient baseline, DistilBERT benefits from contextual language understanding and is therefore better equipped for nuanced text analysis.

This project illustrates the progression from classical NLP methods to modern transformer-based approaches and highlights the importance of model choice in natural language processing tasks.

## Limitations
- Only a subset of the dataset was used
- The transformer model was not fine-tuned
- The comparison focuses primarily on accuracy
- Computational cost was not measured quantitatively

## Future Work
- Fine-tune DistilBERT on the full IMDB dataset
- Compare with additional transformer architectures such as BERT and RoBERTa
- Include more evaluation metrics such as precision, recall, and F1-score
- Perform deeper qualitative error analysis
